In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import librosa
import tensorflow
import librosa.display
import os

In [ ]:
audio="/content/drive/MyDrive/genres_original"

data=[]
labels=[]
genres=os.listdir(audio)

for i in genres:
    genre_path=os.path.join(audio,i)
    for file_name in os.listdir(genre_path):
        file_path=os.path.join(genre_path, file_name)
        if file_path.endswith((".wav", ".mp3", ".au")):
            try:
                y, sr = librosa.load(file_path, duration=30)

                mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
                mfcc_mean = np.mean(mfcc.T, axis=0)

                chroma = librosa.feature.chroma_stft(y=y, sr=sr)
                chroma_mean = np.mean(chroma.T, axis=0)

                contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
                contrast_mean = np.mean(contrast.T, axis=0)


                features = np.concatenate([mfcc_mean, chroma_mean, contrast_mean])

                data.append(features)
                labels.append(i)
            except Exception as e:
                print("Could not process:", file_path, "Error:", e)


/tmp/ipython-input-1483283872.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, duration=30)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Could not process: /content/drive/MyDrive/genres_original/jazz/jazz.00054.wav Error: 


In [ ]:
X = pd.DataFrame(data)
y = pd.Series(labels)

print(X.shape)
print(y.shape)

(359, 32)
(359,)


In [ ]:

en=LabelEncoder()
y_encoded=en.fit_transform(y)

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(X,y_encoded,test_size=0.2,random_state=42)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(x_train)
X_test = scaler.transform(x_test)

In [ ]:
pip install catboost

In [ ]:
from catboost import CatBoostClassifier
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6
)
model.fit(x_train,y_train)

In [ ]:
from sklearn.metrics import accuracy_score
pred=model.predict(x_test)
print(model.score(x_train,y_train))
print(model.score(x_test,y_test))
print(accuracy_score(y_test,pred))

1.0
0.7777777777777778
0.7777777777777778


In [ ]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=300, random_state=42)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.75
